# Activation Checkpointing — Save Memory for Large Models

**What you'll learn:**
- What activations are and why they dominate GPU memory during training
- How `torch.utils.checkpoint.checkpoint` trades compute for memory
- Using `checkpoint_sequential` for sequential models
- Selective Activation Checkpointing (SAC) with `CheckpointPolicy`
- Per-op policies: save expensive ops, recompute cheap ones
- When to use checkpointing (and when not to)

**Prerequisites:** Basic understanding of `nn.Module`, training loops, and backpropagation.

**All code runs on CPU** — patterns transfer directly to GPU.

In [ ]:
import torch
import torch.nn as nn
import torch.utils.checkpoint as cp
import time

print(f"PyTorch version: {torch.__version__}")
print(f"Device: CPU (all examples run on CPU)")

## What Are Activations?

During the **forward pass**, each layer produces intermediate results called **activations**. These are stored in memory because backpropagation needs them to compute gradients.

```
Input → [Layer 1] → act₁ → [Layer 2] → act₂ → ... → [Layer N] → actₙ → Loss
                     ↑ stored          ↑ stored              ↑ stored
```

**The memory problem:** For a model with N layers, all N activations must be kept simultaneously during training. For large models (GPT, ViT), activations dominate memory — often 10-100x the model weights.

**The solution:** Activation checkpointing discards some activations during forward and **recomputes** them during backward, trading ~33% more compute for ~50-90% memory savings.

## Memory Diagram: Without vs With Checkpointing

**Without checkpointing** (standard training):
```
Forward:  [save act₁] [save act₂] [save act₃] [save act₄]  → Peak: 4 activations
Backward: [use act₄]  [use act₃]  [use act₂]  [use act₁]   → Gradients computed
```

**With checkpointing** (checkpoint every 2 layers):
```
Forward:  [save act₂] [discard act₁,₃]  → Peak: 2 activations (saved checkpoints)
Backward: [recompute act₃, act₄ from act₂] → then compute gradients
          [recompute act₁ from input]       → then compute gradients
```

The tradeoff: **~2x less memory, ~1.33x more compute** (one extra forward per segment).

## Basic Checkpoint Usage

`torch.utils.checkpoint.checkpoint(function, *args)` runs `function(*args)` but doesn't save intermediate activations — it recomputes them during backward.

In [ ]:
# A simple multi-layer block
class HeavyBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.linear1 = nn.Linear(dim, dim * 4)
        self.linear2 = nn.Linear(dim * 4, dim)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        return self.norm(self.linear2(torch.relu(self.linear1(x))) + x)

# Use checkpoint on a single block
dim = 256
block = HeavyBlock(dim)
x = torch.randn(8, 32, dim, requires_grad=True)

# Without checkpointing (normal forward)
out_normal = block(x)

# With checkpointing — use_reentrant=False is the modern recommended API
out_ckpt = cp.checkpoint(block, x, use_reentrant=False)

# Results are identical
print(f"Outputs match: {torch.allclose(out_normal, out_ckpt, atol=1e-6)}")
print(f"Output shape: {out_ckpt.shape}")

## Verifying Gradients Match

The key guarantee: checkpointing produces **mathematically identical** gradients. Let's verify this.

In [ ]:
# Compare gradients with and without checkpointing
model = nn.Sequential(*[HeavyBlock(dim) for _ in range(6)])

# Run without checkpointing
x1 = torch.randn(4, 16, dim)
x1.requires_grad_(True)
loss1 = model(x1).sum()
loss1.backward()
grad_no_ckpt = x1.grad.clone()

# Run with checkpointing on each block
x2 = x1.detach().clone().requires_grad_(True)
out = x2
for block in model:
    out = cp.checkpoint(block, out, use_reentrant=False)
loss2 = out.sum()
loss2.backward()
grad_with_ckpt = x2.grad.clone()

print(f"Gradients match: {torch.allclose(grad_no_ckpt, grad_with_ckpt, atol=1e-5)}")
print(f"Max gradient difference: {(grad_no_ckpt - grad_with_ckpt).abs().max().item():.2e}")

## Benchmarking: Compute Overhead

Checkpointing adds ~33% compute overhead because segments are recomputed during backward. Let's measure this on CPU.

In [ ]:
def benchmark_training_step(model, x, use_checkpoint=False, n_iters=5):
    """Time a full forward+backward pass."""
    times = []
    for _ in range(n_iters):
        x_in = x.detach().clone().requires_grad_(True)
        start = time.perf_counter()
        if use_checkpoint:
            out = x_in
            for block in model:
                out = cp.checkpoint(block, out, use_reentrant=False)
        else:
            out = model(x_in)
        loss = out.sum()
        loss.backward()
        times.append(time.perf_counter() - start)
    return sum(times[1:]) / (n_iters - 1)  # skip warmup

big_model = nn.Sequential(*[HeavyBlock(256) for _ in range(12)])
x = torch.randn(8, 32, 256)

time_normal = benchmark_training_step(big_model, x, use_checkpoint=False)
time_ckpt = benchmark_training_step(big_model, x, use_checkpoint=True)

print(f"Without checkpointing: {time_normal*1000:.1f} ms")
print(f"With checkpointing:    {time_ckpt*1000:.1f} ms")
print(f"Overhead: {(time_ckpt/time_normal - 1)*100:.1f}%")
print(f"\n(On GPU, memory savings would be ~50-70% for this 12-layer model)")

## checkpoint_sequential — Easy API for Sequential Models

For models built with `nn.Sequential`, use `checkpoint_sequential` which automatically splits the model into segments and checkpoints each one.

In [ ]:
from torch.utils.checkpoint import checkpoint_sequential

# Build a 12-layer sequential model
seq_model = nn.Sequential(*[HeavyBlock(256) for _ in range(12)])
x = torch.randn(4, 16, 256, requires_grad=True)

# Split into 4 segments (each segment = 3 layers)
# Only segment boundaries store activations
segments = 4
out = checkpoint_sequential(seq_model, segments, x, use_reentrant=False)
loss = out.sum()
loss.backward()

print(f"checkpoint_sequential with {segments} segments")
print(f"  Model layers: {len(seq_model)}")
print(f"  Layers per segment: {len(seq_model) // segments}")
print(f"  Output shape: {out.shape}")
print(f"  Gradient computed: {x.grad is not None}")
print(f"\nMemory: stores {segments} activations instead of {len(seq_model)}")

## Selective Activation Checkpointing (SAC)

Standard checkpointing recomputes **everything** in a segment. But not all operations are equal:
- **Expensive ops** (matmul, conv): slow to recompute, worth saving
- **Cheap ops** (relu, dropout, add): fast to recompute, not worth saving

**SAC** lets you define a **policy** that decides per-operation what to save vs recompute.

In [ ]:
from torch.utils.checkpoint import (
    checkpoint,
    create_selective_checkpoint_contexts,
    CheckpointPolicy,
)

# CheckpointPolicy options:
for policy in CheckpointPolicy:
    print(f"  {policy.name}: {policy.value}")

print("\nKey policies:")
print("  MUST_SAVE: Always save this op's output (expensive ops)")
print("  PREFER_RECOMPUTE: Recompute this op during backward (cheap ops)")

## Writing a Custom Policy Function

A policy function receives an `OpInfo` describing each operation and returns a `CheckpointPolicy`. The idea: save results of expensive ops (like matrix multiplies), recompute cheap ones (like activations).

In [ ]:
from torch.utils.checkpoint import SAC_IGNORED_OPS

# Ops that are expensive to compute (save their outputs)
EXPENSIVE_OPS = {
    torch.ops.aten.mm.default,
    torch.ops.aten.addmm.default,
    torch.ops.aten.bmm.default,
    torch.ops.aten.linear.default,
}

def selective_policy(ctx, op, *args, **kwargs):
    """Save matmul outputs, recompute everything else."""
    if op in SAC_IGNORED_OPS:
        return CheckpointPolicy.MUST_SAVE
    if op in EXPENSIVE_OPS:
        return CheckpointPolicy.MUST_SAVE
    return CheckpointPolicy.PREFER_RECOMPUTE

print("Policy: save matmul/linear outputs, recompute relu/norm/add")
print(f"Expensive ops tracked: {[str(op).split('.')[-2] for op in EXPENSIVE_OPS]}")

## Using SAC with create_selective_checkpoint_contexts

The `context_fn` argument to `checkpoint()` lets you pass a custom context that implements your policy.

In [ ]:
# Use SAC with our policy
block = HeavyBlock(256)
x = torch.randn(4, 16, 256, requires_grad=True)

# context_fn returns (forward_context, backward_context)
context_fn = lambda: create_selective_checkpoint_contexts(selective_policy)

out = checkpoint(
    block,
    x,
    use_reentrant=False,
    context_fn=context_fn,
)
loss = out.sum()
loss.backward()

print(f"SAC forward+backward completed successfully")
print(f"Output shape: {out.shape}")
print(f"Gradient shape: {x.grad.shape}")
print(f"\nWith SAC: matmul outputs are SAVED, relu/norm are RECOMPUTED")
print(f"Best of both worlds: less memory than full save, less compute than full recompute")

## Comparing All Three Strategies

Let's compare: no checkpointing, full checkpointing, and SAC — verifying all produce the same gradients.

In [ ]:
model = nn.Sequential(*[HeavyBlock(128) for _ in range(8)])
x_base = torch.randn(4, 16, 128)

def run_no_checkpoint(model, x):
    x = x.detach().clone().requires_grad_(True)
    loss = model(x).sum()
    loss.backward()
    return x.grad

def run_full_checkpoint(model, x):
    x = x.detach().clone().requires_grad_(True)
    out = x
    for block in model:
        out = cp.checkpoint(block, out, use_reentrant=False)
    out.sum().backward()
    return x.grad

def run_sac(model, x):
    x = x.detach().clone().requires_grad_(True)
    ctx_fn = lambda: create_selective_checkpoint_contexts(selective_policy)
    out = x
    for block in model:
        out = cp.checkpoint(block, out, use_reentrant=False, context_fn=ctx_fn)
    out.sum().backward()
    return x.grad

grad_none = run_no_checkpoint(model, x_base)
grad_full = run_full_checkpoint(model, x_base)
grad_sac = run_sac(model, x_base)

print("Gradient correctness check:")
print(f"  Full ckpt vs no ckpt: {torch.allclose(grad_none, grad_full, atol=1e-5)}")
print(f"  SAC vs no ckpt:       {torch.allclose(grad_none, grad_sac, atol=1e-5)}")
print(f"\nAll three produce identical gradients! ✓")

## Integrating Checkpointing into a Model Class

In practice, you apply checkpointing inside your model's `forward()` method, controlled by a flag.

In [ ]:
class DeepModel(nn.Module):
    def __init__(self, dim, n_layers, use_checkpoint=False):
        super().__init__()
        self.layers = nn.ModuleList([HeavyBlock(dim) for _ in range(n_layers)])
        self.use_checkpoint = use_checkpoint
        self.head = nn.Linear(dim, 10)

    def forward(self, x):
        for layer in self.layers:
            if self.use_checkpoint and self.training:
                x = cp.checkpoint(layer, x, use_reentrant=False)
            else:
                x = layer(x)
        return self.head(x.mean(dim=1))  # pool + classify

# Demonstrate the pattern
model_ckpt = DeepModel(128, n_layers=8, use_checkpoint=True)
model_ckpt.train()

x = torch.randn(4, 32, 128)
logits = model_ckpt(x)
loss = logits.sum()
loss.backward()

print(f"Model with checkpointing — forward+backward OK")
print(f"  Layers: 8, Checkpointed: {model_ckpt.use_checkpoint}")
print(f"  Output: {logits.shape}")

# In eval mode, checkpointing is skipped (no need to save memory at inference)
model_ckpt.eval()
with torch.no_grad():
    logits_eval = model_ckpt(x)
print(f"\n  Eval mode runs without checkpointing (faster inference)")

## When to Use Checkpointing — Guidelines

| Scenario | Recommendation |
|----------|---------------|
| Model fits in memory comfortably | Don't checkpoint — unnecessary overhead |
| OOM during training | Checkpoint every layer |
| Need max throughput + tight memory | Use SAC — save matmuls, recompute cheap ops |
| Very deep model (50+ layers) | Use `checkpoint_sequential` with 4-8 segments |
| Inference only | Never — checkpointing only helps backward pass |
| Mixed precision (AMP) training | Checkpointing works with AMP, use together |
| Gradient accumulation | Combine: smaller micro-batches + checkpointing |

**Rule of thumb:** If you're within 20% of OOM, try checkpointing before reducing batch size.

## Important: use_reentrant=False

Always use `use_reentrant=False` (the modern API). The legacy `use_reentrant=True` has subtle bugs with:
- `torch.autograd.grad()`
- Multiple `backward()` calls
- Nested checkpointing

The non-reentrant version is correct in all cases.

In [ ]:
# Demonstrating that use_reentrant=False works with torch.autograd.grad
model = HeavyBlock(64)
x = torch.randn(2, 8, 64, requires_grad=True)

out = cp.checkpoint(model, x, use_reentrant=False)
loss = out.sum()

# This works correctly with use_reentrant=False
grad = torch.autograd.grad(loss, x, create_graph=True)[0]
print(f"torch.autograd.grad works with non-reentrant checkpoint: grad shape = {grad.shape}")

# Higher-order gradients also work
grad_of_grad = torch.autograd.grad(grad.sum(), x)[0]
print(f"Second-order gradients also work: shape = {grad_of_grad.shape}")

## Try It Yourself

**Exercise:** Add checkpointing to a 12-layer model and compare the training step time.

1. Create a model with 12 `HeavyBlock` layers (dim=512 for a noticeable difference)
2. Run a forward+backward pass WITHOUT checkpointing, time it
3. Run a forward+backward pass WITH checkpointing on each layer, time it
4. Try `checkpoint_sequential` with different segment counts (2, 4, 6) — which is fastest?

In [ ]:
# YOUR CODE HERE — Exercise solution scaffold
dim = 512
n_layers = 12

# Step 1: Build the model
exercise_model = nn.Sequential(*[HeavyBlock(dim) for _ in range(n_layers)])
x = torch.randn(4, 16, dim)

# Step 2: Time without checkpointing
# start = time.perf_counter()
# ... your code ...
# print(f"No checkpoint: {elapsed:.1f}ms")

# Step 3: Time with per-layer checkpointing
# ... your code ...

# Step 4: Try checkpoint_sequential with segments=2, 4, 6
# for segs in [2, 4, 6]:
#     ... your code ...

print("Uncomment the code above and fill in the blanks!")

## Key Takeaways

1. **Activation checkpointing** trades compute for memory — recomputes activations during backward instead of storing them
2. **`torch.utils.checkpoint.checkpoint(fn, *args, use_reentrant=False)`** is the basic API
3. **`checkpoint_sequential`** is a convenience for `nn.Sequential` models — just specify number of segments
4. **Selective Activation Checkpointing (SAC)** gives fine-grained control: save expensive ops (matmul), recompute cheap ones (relu)
5. **Always use `use_reentrant=False`** — the reentrant version has known correctness issues
6. **Gradients are identical** with or without checkpointing — it's a pure memory optimization
7. **Typical overhead: 20-40%** more compute, **50-90%** less activation memory
8. **Don't checkpoint at inference** — only useful during training (backward pass)
9. **Combine with other techniques:** gradient accumulation, mixed precision, FSDP for maximum efficiency